In [0]:
df = spark.read.csv("/Volumes/workspace/bronze/source_csv_files/Coffe_sales.csv", header=True, inferSchema=True)
display(df)

In [0]:
from pyspark.sql.functions import monotonically_increasing_id, when

df1 = df.withColumn("transaction_id", monotonically_increasing_id() + 1)
df1 = df1.withColumn(
    "cash_type",
    when(df1.money < 30, "cash")
    .when(df1.money > 35, "upi")
    .otherwise(df1.cash_type)
)
cols = ["transaction_id"] + [col for col in df1.columns if col != "transaction_id"]
df1 = df1.select(*cols)
display(df1)

In [0]:
df1.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("coffee_name", "cash_type") \
    .option("path", "/Volumes/workspace/silver/coffe_sales") \
    .save()

In [0]:
df_americano = spark.read.format("delta").load("/Volumes/workspace/silver/coffe_sales")
df_americano = df_americano.filter(df_americano.coffee_name == "Americano with Milk")
display(df_americano)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy("cash_type", "time_of_day").orderBy("transaction_id")
df_americano = df_americano.withColumn("row_num", row_number().over(window_spec))

selected_cols = [df_americano.columns[i] for i in [0, 2, 3, 4, 5, 10]] + ["row_num"]
df_americano_selected = df_americano.select(*selected_cols)
display(df_americano_selected)

In [0]:
from pyspark.sql.functions import sum, avg, count

window_spec = Window.partitionBy("cash_type", "time_of_day")

df_americano_agg = df_americano.withColumn("sum_money", sum("money").over(window_spec)) \
    .withColumn("avg_money", avg("money").over(window_spec)) \
    .withColumn("count_transaction_type", count("money").over(window_spec))

display(df_americano_agg)

In [0]:
from pyspark.sql.functions import sum, avg, count, round

window_spec = Window.partitionBy("cash_type", "time_of_day")

df_americano_agg = df_americano.withColumn("sum_money", round(sum("money").over(window_spec), 2)) \
    .withColumn("avg_money", round(avg("money").over(window_spec), 2)) \
    .withColumn("count_transaction_type", count("money").over(window_spec))

display(df_americano_agg)

In [0]:
from pyspark.sql.functions import sum, avg, round

df_gold = df_americano.groupBy("cash_type", "time_of_day") \
    .agg(
        round(sum("money"), 2).alias("sum_money"),
        round(avg("money"), 2).alias("avg_money")
    )

df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("path", "/Volumes/workspace/gold/final_coffe_sales_report").save()

spark.sql("""
OPTIMIZE delta.`/Volumes/workspace/gold/final_coffe_sales_report`
ZORDER BY (cash_type)
""")

In [0]:
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.final_coffe_sales_report")

In [0]:
%sql
select * from workspace.gold.final_coffe_sales_report

In [0]:
dbutils.widgets.dropdown(
    "coffee_name",
    "Americano with Milk",
    [row.coffee_name for row in spark.read.format("delta").load("/Volumes/workspace/silver/coffe_sales").select("coffee_name").distinct().collect()]
)

selected_coffee = dbutils.widgets.get("coffee_name")
df_americano = spark.read.format("delta").load("/Volumes/workspace/silver/coffe_sales")
df_americano = df_americano.filter(df_americano.coffee_name == selected_coffee)
display(df_americano)

In [0]:
system.lakeflow.job_run_timeline

In [0]:
%sql
select * from workspace.bronze1.coffesales

In [0]:
spark.sql("""
DELETE FROM workspace.bronze1.coffesales WHERE transaction_id < 10
""")